In [1]:
import pandas as pd 
import numpy as np
import duckdb 
import logging 


logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    filename='quant-of-uncertainty.log'
)
logger = logging.getLogger(__name__)

In [4]:
### load duckdb tables into pandas dfs 
con = None 
try: 
    # create and verify connection 
    con = duckdb.connect(database='project1.db', read_only=False) 
    logger.info("Connected to duckdb instance.") 

    # inserting tables 
    cville = con.execute(f"""
        SELECT * FROM cville;
    """).fetchdf()
    dulles = con.execute(f"""
        SELECT * FROM dulles;
    """).fetchdf()
    lynchburg = con.execute(f"""
        SELECT * FROM lynchburg;
    """).fetchdf()
    norfolk = con.execute(f"""
        SELECT * FROM norfolk;
    """).fetchdf()

    logger.info("All tables loaded into pandas dataframes")

except Exception as e:
    logger.error(f"An error occurred: {e}")

finally:
    if con:
        con.close()
        logger.info("Duckdb connection closed.")

In [5]:
# Define seasons
seasons = {
    "Winter": [12, 1, 2],
    "Spring": [3, 4, 5],
    "Summer": [6, 7, 8],
    "Fall":   [9, 10, 11]
}

# Suppose cleaned dataframes are in a dict like before
dfs = {
    "cville": cville,
    "dulles": dulles,
    "norfolk": norfolk,
    "lynchburg": lynchburg
}

# Dictionary to store std for each city and season
std_by_city_season = {}

for city_name, df in dfs.items():
    df = df.copy()
    
    # Ensure Date column is datetime
    df['DATE'] = pd.to_datetime(df['DATE'])
    
    # Extract month
    df['Month'] = df['DATE'].dt.month
    
    # Dictionary for this city
    city_std = {}
    
    for season_name, months in seasons.items():
        # Filter rows belonging to this season
        season_df = df[df['Month'].isin(months)]
        
        # Compute std of temperature for this season
        city_std[season_name] = season_df['DailyAverageDryBulbTemperature'].std()
    
    std_by_city_season[city_name] = city_std

# Print nicely
for city, season_std in std_by_city_season.items():
    print(f"\n{city}:")
    for season, std in season_std.items():
        print(f"  {season}: {std:.2f}")
        
logging.info("Calculations completed and results printed.")


cville:
  Winter: 9.82
  Spring: 11.82
  Summer: 5.38
  Fall: 11.91

dulles:
  Winter: 9.93
  Spring: 12.32
  Summer: 5.30
  Fall: 12.41

norfolk:
  Winter: 10.09
  Spring: 11.46
  Summer: 4.79
  Fall: 11.25

lynchburg:
  Winter: 9.58
  Spring: 11.34
  Summer: 4.74
  Fall: 11.87
